# AutoMod AI: car-parts segmenter
Trains a small YOLO segmentation model on the public Carparts-seg dataset (23 classes: hood, wheel, glass, lights, doors, bumpers, trunk, mirrors). The inference server uses it to edit only the chosen part.

**Runtime:** Runtime > Change runtime type > T4 GPU. Then Runtime > Run all. About 30 to 60 minutes.

In [ ]:
!pip -q install ultralytics
import torch
print('cuda:', torch.cuda.is_available())

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11s-seg.pt')
results = model.train(
    data='carparts-seg.yaml',
    epochs=60,
    imgsz=640,
    batch=16,
    project='runs',
    name='carparts',
    exist_ok=True,
    patience=15,
)

In [ ]:
best = YOLO('runs/carparts/weights/best.pt')
m = best.val(data='carparts-seg.yaml', imgsz=640)
print('mask mAP50:', m.seg.map50, ' mask mAP50-95:', m.seg.map)
print(best.names)

In [ ]:
import glob, os
from IPython.display import Image, display
for f in ['runs/carparts/results.png', 'runs/carparts/confusion_matrix_normalized.png', 'runs/carparts/val_batch0_pred.jpg']:
    if os.path.exists(f):
        display(Image(f))

In [ ]:
import shutil
shutil.copy('runs/carparts/weights/best.pt', 'automod_parts_seg.pt')
from google.colab import files
files.download('automod_parts_seg.pt')
files.download('runs/carparts/results.png')
if os.path.exists('runs/carparts/confusion_matrix_normalized.png'):
    files.download('runs/carparts/confusion_matrix_normalized.png')